# 03 — Mini Pipeline (Evening 3)

**What this builds:**
3-node LangGraph pipeline — `universe_filter` → `technical_node` → `display_node`

**The key question:** Do the candidates look like stocks you would actually consider trading?

**Cost: ~₹80** (~15 LLM calls)

In [ ]:
# Cell 1 — All imports
import warnings; warnings.filterwarnings('ignore')
from dotenv import load_dotenv
import os, time
from datetime import date
from typing import Optional, Literal

import pandas as pd
import pandas_ta as ta
import yfinance as yf
import plotly.graph_objects as go
from IPython.display import display, HTML

from pydantic import BaseModel, Field, field_validator
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict

load_dotenv('../.env') or load_dotenv('.env')
AS_OF_DATE = date.today().isoformat()
print(f'as_of_date: {AS_OF_DATE}')

In [ ]:
# Cell 2 — Data + indicator functions (same as notebook 02)
def get_ohlcv(symbol, as_of_date):
    ticker = f'{symbol}.NS' if symbol != '^CNX500' else symbol
    raw = yf.download(ticker, period='max', auto_adjust=True, progress=False)
    if raw.empty: raise ValueError(f'No data for {ticker}')
    if isinstance(raw.columns, pd.MultiIndex): raw.columns = raw.columns.droplevel(1)
    raw.columns = [c.lower().replace(' ','_') for c in raw.columns]
    raw.index = pd.to_datetime(raw.index).normalize().date
    cutoff = date.fromisoformat(as_of_date)
    df = raw[raw.index <= cutoff].copy()
    if len(df) < 100: raise ValueError(f'{symbol} only {len(df)} rows')
    return df

def compute_indicators(df):
    df = df.copy()
    df['ema_20']     = ta.ema(df['close'], length=20)
    df['ema_50']     = ta.ema(df['close'], length=50)
    df['ema_200']    = ta.ema(df['close'], length=200)
    df['atr_14']     = ta.atr(df['high'], df['low'], df['close'], length=14)
    df['rsi_14']     = ta.rsi(df['close'], length=14)
    bb = ta.bbands(df['close'], length=20, std=2.0)
    if bb is not None:
        df['bb_upper'] = bb['BBU_20_2.0']
        df['bb_lower'] = bb['BBL_20_2.0']
    df['vol_avg_20'] = df['volume'].rolling(20).mean()
    return df

def identify_swings(df, pct=0.03):
    highs, lows = [], []
    direction, last_pivot = None, float(df['close'].iloc[0])
    for i in range(1, len(df)):
        h, l = float(df['high'].iloc[i]), float(df['low'].iloc[i])
        if direction != 'down' and h >= last_pivot*(1+pct):
            highs.append(round(h,2)); last_pivot=h; direction='down'
        elif direction != 'up' and l <= last_pivot*(1-pct):
            lows.append(round(l,2));  last_pivot=l; direction='up'
    lc = float(df['close'].iloc[-1])
    return {
        'nearest_support':    max([l for l in lows  if l<lc], default=round(float(df['low'].min()),2)),
        'nearest_resistance': min([h for h in highs if h>lc], default=round(float(df['high'].max()),2)),
        'recent_highs': sorted(highs,reverse=True)[:5],
        'recent_lows':  sorted(lows, reverse=True)[:5],
    }

def relative_strength(symbol, as_of_date, weeks=13):
    wd = weeks*5
    s = get_ohlcv(symbol, as_of_date)
    n = get_ohlcv('^CNX500', as_of_date)
    if len(s)<wd or len(n)<wd: return 50.0
    sr = s['close'].iloc[-1]/s['close'].iloc[-wd]-1
    nr = n['close'].iloc[-1]/n['close'].iloc[-wd]-1
    ratio = (1+sr)/(1+nr)
    return round(min(100.0,max(0.0,(ratio-0.7)/0.6*100)),1)

print('✓ Data functions loaded')

In [ ]:
# Cell 3 — TechnicalScorecard + BACKSTORY (same as notebook 02)
BACKSTORY_VERSION = 'v1.0'

class TechnicalScorecard(BaseModel):
    symbol: str; as_of_date: str
    setup_type: Literal['pullback_uptrend','breakout_base','mean_reversion_range','rs_leader','none']
    direction:  Literal['long','short','none']
    weekly_trend: Literal['up','down','sideways']
    daily_trend:  Literal['up','down','sideways']
    component_scores: dict
    technical_score: int = Field(ge=0, le=100)
    rs_rating_pct:   float = Field(ge=0, le=100)
    rr_ratio:        float
    entry_zone:   tuple[float, float]
    initial_stop: float
    target_1:     float
    target_2:     Optional[float] = None
    score_reasoning: str
    volume_signal: Literal['expansion','neutral','contraction']
    rejected:         bool = False
    rejection_reason: Optional[str] = None

    @field_validator('component_scores')
    @classmethod
    def check_components(cls, v):
        required = {'trend','structure','setup','volume'}
        if set(v.keys()) != required:
            raise ValueError(f'component_scores must have exactly: {required}')
        return v

TECHNICAL_BACKSTORY = """
You are a chart technician who has analysed Indian equity charts for 15 years.
You think weekly trend first, daily structure second, trigger setup third.
You believe price + volume + structure beat every indicator combination.
You use indicators only as confirmation, never as primary signal.
You know that Indian retail traders fail because they:
1. Trade against the weekly trend
2. Enter at the top of a daily move chasing momentum
3. Ignore relative strength and pick laggards
4. Use round-number stops that get hunted

YOU CLASSIFY EXACTLY FOUR SETUP TYPES:
1. pullback_uptrend: weekly trend UP, price above EMA-200, pullback to rising EMA-20, RSI 40-55
2. breakout_base: 6-12 week consolidation, breakout with volume >= 1.5x average
3. mean_reversion_range: horizontal range 4+ weeks, price at range boundary
4. rs_leader: RS above 75th percentile + any of the above 3 setups

SCORING (4 components, total 0-100):
trend(0-30), structure(0-25), setup(0-30), volume(0-15)

REFUSAL RULES:
- Reject if weekly trend does not agree with setup direction
- Reject if price below EMA-200 for long setups
- Reject if RSI above 75 at entry for longs
- Reject if stop > 8% from entry
- Reject if R:R < 1.8
- Reject if chart is choppy with no clear structure
- Stops must be at real structural levels from the provided swing data — never round numbers
"""

def build_prompt(symbol, as_of_date, df, swings, rs):
    last = df.iloc[-1]
    ema200_slope = (float(df['ema_200'].iloc[-1]) - float(df['ema_200'].iloc[-10]))
    vol_ratio = float(last['volume']) / float(last['vol_avg_20'])
    return f"""Analyse {symbol} as of {as_of_date}.
PRICE: ₹{float(last['close']):.2f} | EMA-20: ₹{float(last['ema_20']):.2f} ({'ABOVE' if last['close']>last['ema_20'] else 'BELOW'}) | EMA-50: ₹{float(last['ema_50']):.2f} | EMA-200: ₹{float(last['ema_200']):.2f} ({'ABOVE' if last['close']>last['ema_200'] else 'BELOW'})
EMA-200 slope: {'rising' if ema200_slope > 0 else 'falling'}
ATR(14): ₹{float(last['atr_14']):.2f} ({float(last['atr_14'])/float(last['close'])*100:.1f}%) | RSI: {float(last['rsi_14']):.1f} | Volume: {vol_ratio*100:.0f}% of avg
SWING LEVELS — nearest support: ₹{swings['nearest_support']} | resistance: ₹{swings['nearest_resistance']}
Recent lows:  {swings['recent_lows']}
Recent highs: {swings['recent_highs']}
RS vs Nifty500 (13w): {rs:.1f}th percentile
Place stop at nearest structural swing low from the data. No round numbers."""

print('✓ Schema and backstory ready')

## The 3-node pipeline

```
universe_filter_node  →  technical_node  →  display_node  →  END
     (pure Python)         (LLM call)        (no LLM)
     top 5 by RS            score each        show results
```


In [ ]:
# Cell 4 — PipelineState (expanded for 3 nodes)
class PipelineState(TypedDict):
    run_date:   str
    all_stocks: list       # input: 10 hardcoded stocks
    universe:   list       # after RS filter: top 5
    scorecards: list       # TechnicalScorecard list
    errors:     list
    start_time: float

print('✓ PipelineState defined')

In [ ]:
# Cell 5 — Node 1: universe_filter (pure Python, no LLM)
# Ranks stocks by 13-week RS, returns top 5
# Spec ref: §4 Agent 1 Universe Curator role (simplified version)

def universe_filter_node(state: PipelineState) -> PipelineState:
    """Filter 10 stocks to top 5 by relative strength. No LLM."""
    print('\n[Node 1] Universe filter — ranking by RS...')
    rs_scores = {}

    for sym in state['all_stocks']:
        try:
            rs = relative_strength(sym, state['run_date'])
            rs_scores[sym] = rs
            print(f'  {sym:<12} RS: {rs:.0f}th percentile')
        except Exception as e:
            print(f'  {sym:<12} error: {e}')
            rs_scores[sym] = 0.0

    # Top 5 by RS
    top5 = sorted(rs_scores, key=rs_scores.get, reverse=True)[:5]
    print(f'\n  → Top 5: {top5}')

    return {**state, 'universe': top5}


print('✓ universe_filter_node defined')

In [ ]:
# Cell 6 — Node 2: technical_node (LLM call per stock)

def technical_node(state: PipelineState) -> PipelineState:
    """Run Technical Analyst on each stock in universe."""
    print('\n[Node 2] Technical Analyst — classifying setups...')
    llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0, seed=42)
    structured_llm = llm.with_structured_output(TechnicalScorecard)
    scorecards = []

    for sym in state['universe']:
        try:
            df     = get_ohlcv(sym, state['run_date'])
            df     = compute_indicators(df)
            swings = identify_swings(df)
            rs     = relative_strength(sym, state['run_date'])
            prompt = build_prompt(sym, state['run_date'], df, swings, rs)

            sc = structured_llm.invoke([
                SystemMessage(content=TECHNICAL_BACKSTORY),
                HumanMessage(content=prompt),
            ])
            # Override RS with Python-computed value
            sc_dict = sc.model_dump()
            sc_dict['rs_rating_pct'] = rs
            sc = TechnicalScorecard(**sc_dict)
            scorecards.append(sc)

            status = sc.setup_type if not sc.rejected else f'rejected ({sc.rejection_reason[:40]})'
            print(f'  {sym:<12} {status}  score={sc.technical_score}')

        except Exception as e:
            err = f'technical/{sym}: {e}'
            print(f'  {sym:<12} ERROR: {e}')
            state['errors'].append(err)

    return {**state, 'scorecards': scorecards}


print('✓ technical_node defined')

In [ ]:
# Cell 7 — Node 3: display_node (no LLM — pure display)

def display_node(state: PipelineState) -> PipelineState:
    """Render results. No LLM calls."""
    elapsed = time.time() - state['start_time']
    scorecards = state.get('scorecards', [])
    candidates = [sc for sc in scorecards if not sc.rejected]
    rejected   = [sc for sc in scorecards if sc.rejected]

    print(f'\n[Node 3] Display — {len(candidates)} candidates, {len(rejected)} rejected')
    print(f'  Pipeline time: {elapsed:.0f}s')

    if state['errors']:
        print(f'  Errors: {state["errors"]}')

    # Header
    display(HTML(f"""
    <div style="background:#1e1e2e;padding:12px 16px;border-radius:8px;margin:10px 0">
      <h2 style="color:#cdd6f4;margin:0">SwingTrader Mini-Briefing — {state['run_date']}</h2>
      <p style="color:#585b70;margin:4px 0 0">
        {len(candidates)} candidates · {len(rejected)} rejected · {elapsed:.0f}s runtime
      </p>
    </div>
    """))

    if not candidates:
        display(HTML("""
        <div style="background:#1e1e2e;border-left:3px solid #f9a825;
                    padding:10px 14px;color:#f9a825;margin:8px 0">
          No candidates today — all stocks rejected by Technical Analyst.
          Check market conditions or loosen backstory refusal rules.
        </div>
        """))
        return state

    # Sort by score
    candidates.sort(key=lambda sc: sc.technical_score, reverse=True)

    for rank, sc in enumerate(candidates, 1):
        stop_pct = abs(sc.initial_stop - sc.entry_zone[0]) / sc.entry_zone[0] * 100
        display(HTML(f"""
        <div style="background:#1e1e2e;border-radius:8px;padding:12px 16px;
                    margin:8px 0;border-left:3px solid #a6e3a1">
          <div style="display:flex;justify-content:space-between;align-items:center">
            <span style="color:#cdd6f4;font-size:16px;font-weight:700">
              #{rank} &nbsp; {sc.symbol}
            </span>
            <span style="background:#a6e3a1;color:#1e1e2e;padding:2px 10px;
                         border-radius:20px;font-weight:700;font-size:13px">
              {sc.technical_score}
            </span>
          </div>
          <div style="color:#585b70;font-size:12px;margin:4px 0">
            {sc.setup_type} · {sc.direction} · RS: {sc.rs_rating_pct:.0f}th pct
          </div>
          <div style="display:grid;grid-template-columns:repeat(4,1fr);
                      gap:8px;margin-top:8px">
            <div style="background:#181825;padding:6px 8px;border-radius:6px">
              <div style="color:#585b70;font-size:10px">ENTRY</div>
              <div style="color:#cdd6f4;font-size:13px;font-weight:600">
                ₹{sc.entry_zone[0]:.0f}–{sc.entry_zone[1]:.0f}
              </div>
            </div>
            <div style="background:#181825;padding:6px 8px;border-radius:6px">
              <div style="color:#585b70;font-size:10px">STOP</div>
              <div style="color:#f38ba8;font-size:13px;font-weight:600">
                ₹{sc.initial_stop:.0f} (-{stop_pct:.1f}%)
              </div>
            </div>
            <div style="background:#181825;padding:6px 8px;border-radius:6px">
              <div style="color:#585b70;font-size:10px">TARGET 1</div>
              <div style="color:#a6e3a1;font-size:13px;font-weight:600">
                ₹{sc.target_1:.0f}
              </div>
            </div>
            <div style="background:#181825;padding:6px 8px;border-radius:6px">
              <div style="color:#585b70;font-size:10px">R:R</div>
              <div style="color:#89b4fa;font-size:13px;font-weight:600">
                {sc.rr_ratio:.1f}x
              </div>
            </div>
          </div>
          <div style="color:#585b70;font-size:11px;margin-top:8px;font-style:italic">
            {sc.score_reasoning}
          </div>
        </div>
        """))

    # Rejected summary
    if rejected:
        rejected_html = ''.join(
            f'<li style="color:#585b70">{sc.symbol}: {sc.rejection_reason}</li>'
            for sc in rejected
        )
        display(HTML(f"""
        <div style="background:#1e1e2e;padding:10px 14px;border-radius:6px;margin-top:8px">
          <p style="color:#585b70;font-size:12px;margin:0 0 6px">Rejected:</p>
          <ul style="margin:0;padding-left:20px">{rejected_html}</ul>
        </div>
        """))

    return state


print('✓ display_node defined')

In [ ]:
# Cell 8 — Conditional edge: skip if no universe candidates
def route_after_universe(state: PipelineState) -> str:
    if not state.get('universe'):
        print('  → No stocks in universe — stopping early')
        return 'display'   # go to display which will show no-trade message
    return 'technical'

def route_after_technical(state: PipelineState) -> str:
    candidates = [sc for sc in state.get('scorecards', []) if not sc.rejected]
    if not candidates:
        print('  → All stocks rejected — going to display')
    return 'display'


# Wire the graph
graph = StateGraph(PipelineState)
graph.add_node('universe',  universe_filter_node)
graph.add_node('technical', technical_node)
graph.add_node('display',   display_node)

graph.set_entry_point('universe')
graph.add_conditional_edges('universe',  route_after_universe,
                            {'technical': 'technical', 'display': 'display'})
graph.add_conditional_edges('technical', route_after_technical,
                            {'display': 'display'})
graph.add_edge('display', END)

app = graph.compile()
print('✓ 3-node pipeline compiled')
print('  Nodes:', list(app.get_graph().nodes.keys()))

In [ ]:
# Cell 9 — DEFINE the 10-stock universe and RUN the pipeline
# Mix of sectors — IT, pharma, banking, infra, FMCG, energy
ALL_STOCKS = [
    'INFY',        # Nifty IT
    'DRREDDY',     # Nifty Pharma
    'HDFCBANK',    # Nifty Banking
    'LT',          # Nifty Infra
    'HINDUNILVR',  # Nifty FMCG
    'RELIANCE',    # Large cap conglomerate
    'SUNPHARMA',   # Pharma 2
    'BAJFINANCE',  # NBFC
    'TATASTEEL',   # Metals
    'WIPRO',       # IT 2
]

initial_state = {
    'run_date':   AS_OF_DATE,
    'all_stocks': ALL_STOCKS,
    'universe':   [],
    'scorecards': [],
    'errors':     [],
    'start_time': time.time(),
}

print(f'Starting 3-node pipeline for {AS_OF_DATE}...')
print(f'Input: {len(ALL_STOCKS)} stocks  →  top 5 by RS  →  technical classification\n')

result = app.invoke(initial_state)

In [ ]:
# Cell 10 — Inspect the full pipeline state
print('Final pipeline state:')
print(f'  run_date:  {result["run_date"]}')
print(f'  universe:  {result["universe"]}  (top 5 by RS)')
print(f'  scorecards: {len(result["scorecards"])} total')
candidates = [sc for sc in result['scorecards'] if not sc.rejected]
rejected   = [sc for sc in result['scorecards'] if sc.rejected]
print(f'    {len(candidates)} candidates  |  {len(rejected)} rejected')
if result['errors']:
    print(f'  errors: {result["errors"]}')
print()
print('Candidate scores:')
for sc in sorted(candidates, key=lambda x: x.technical_score, reverse=True):
    print(f'  {sc.symbol:<12} {sc.setup_type:<25} score={sc.technical_score}')

## Evening 3 Gate

Ask yourself honestly:

- [ ] Does at least 1 candidate look like a real trade you would consider?
- [ ] Do the rejection reasons make sense for the current market?
- [ ] Are stops at structural levels — not round numbers like ₹1,500 or ₹2,000?
- [ ] Is the pipeline cost under ₹80 for 10 stocks? (`result['start_time']` gives you the time)

If yes to all: **the POC proves the concept.**

**Next steps:**
1. Run on a different date to see how output varies with market conditions
2. Add 10 more stocks to the universe for broader coverage
3. Start the full calibration (30+ historical pairs) to measure Spearman rho
4. That leads into `04_evaluate_output.ipynb` and `05_calibration.ipynb`